# 🌐 Assistente Diversa — Educação Inclusiva
## Proposta final - TCC

Grupo: Hélice

Participantes:
- Brena Tudonovo
- Cesar Ricciarelli
- João Pedro Maranhão
- Leandro Nascimento
- Nícolas Pinotti

## 1. Instalação de dependências



In [12]:
%pip install -U beautifulsoup4 gradio pandas scikit-learn requests python-dotenv

print("✅ Instalação concluída!")
print()
print("Bibliotecas usadas no projeto:")
print("  • gradio     — interface web do chatbot")
print("  • requests   — chamadas HTTP para a Groq")
print("  • pandas     — manipulação da base de dados")
print("  • sklearn    — TF-IDF e busca por similaridade")


Note: you may need to restart the kernel to use updated packages.
✅ Instalação concluída!

Bibliotecas usadas no projeto:
  • gradio     — interface web do chatbot
  • requests   — chamadas HTTP para a Groq
  • pandas     — manipulação da base de dados
  • sklearn    — TF-IDF e busca por similaridade


## 2. Importações e configuração



Carregamento da chave de API do GROQ devidamente configurada com proteção através do .env (exluso na configuração de .gitignore).

In [13]:
import json
import os
import re
import time
from typing import Dict, List

import gradio as gr
import pandas as pd
import requests
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from dotenv import load_dotenv

load_dotenv()
GROQ_KEY = os.getenv("GROQ_KEY", "")

GROQ_MODEL = "llama-3.1-8b-instant"
GROQ_URL = "https://api.groq.com/openai/v1/chat/completions"

PARAMS_GERACAO = {
    "model": GROQ_MODEL,
    "max_tokens": 600,
    "temperature": 0.3,
    "top_p": 0.9,
}

print("✅ Configuração carregada")
print("Modelo:", GROQ_MODEL)
print("Modo:", "Groq/Llama" if GROQ_KEY else "Demonstração sem API")

✅ Configuração carregada
Modelo: llama-3.1-8b-instant
Modo: Groq/Llama


## 3. Base de conhecimento


Os dados se encontram hospedados no google docs, onde os registros ficam mais seguros e o código fica mais limpo.

In [14]:
import pandas as pd

SHEET_ID = "1Ll0gKNxMs20rMAESmjk4Dk2_FAtugKgY5NUhwaod1rI" 
URL_PLANILHA = f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/export?format=csv"

# 1. Transforma a planilha em um Pandas DataFrame
artigos_brutos = pd.read_csv(URL_PLANILHA)
artigos_brutos = artigos_brutos.rename(columns={"t": "titulo"})

# 2. Visualizando dimensões do Dataframe
print("Linhas x Colunas =", artigos_brutos.shape)


# 3. Imprime as colunas (corrigido: sem os parênteses no final)
print("\nColunas presentes na base de dados:")
print(list(artigos_brutos.columns))
print()

# 4. Visualiza as 5 primeiras linhas de forma formatada no Jupyter
print("5 primeiras linhas")
display(artigos_brutos.head())

# 5. Visualização dos tipos dos objetos
print(f"Todos os objetos são de texto:")
display(artigos_brutos.dtypes)

Linhas x Colunas = (36, 4)

Colunas presentes na base de dados:
['titulo', 'url', 'texto', 'artigo_completo']

5 primeiras linhas


,titulo,url,texto,artigo_completo
0,TEA na educação infantil: inclusão e afetivida...,https://diversa.org.br/artigos/tea-educacao-in...,O artigo apresenta estratégias que podem apoia...,Renata Pantaleão Ubugata Publicado em 03/10/20...
1,Ferramentas para o desenvolvimento escolar de ...,https://diversa.org.br/artigos/ferramentas-par...,O artigo aborda estratégias para apoiar a apre...,Emanoele Freitas Publicado em 03/04/2023\nEm c...
2,O que é Transtorno do Déficit de Atenção e Hip...,https://diversa.org.br/artigos/o-que-e-transto...,O artigo explica o Transtorno do Déficit de At...,Priscila Romero Publicado em 13/07/2023\nDia 1...
3,Como a escola pode lidar com os diagnósticos m...,https://diversa.org.br/artigos/como-a-escola-p...,O artigo explica o Transtorno do Déficit de At...,Patrícia de Brito Publicado em 19/04/2019\nDe ...
4,Psicomotricidade como recurso para a educação ...,https://diversa.org.br/artigos/psicomotricidad...,O artigo propõe uma reflexão cuidadosa sobre d...,"Caroline Lago, Cristiana Crespo e Lucas Bragan..."


Todos os objetos são de texto:


titulo             str
url                str
texto              str
artigo_completo    str
dtype: object

## 4. Limpeza da base



In [15]:
def limpar_texto(texto):
    texto = re.sub(r"\s+", " ", str(texto))
    texto = re.sub(r"\[.*?\]", "", texto)
    texto = re.sub(r"http\S+", "", texto)
    return texto.strip()

df = pd.DataFrame(artigos_brutos)
df["texto_limpo"] = df["texto"].apply(limpar_texto)
df = df[df["texto_limpo"].str.len() > 120].reset_index(drop=True)
df["conteudo_busca"] = df["titulo"] + ". " + df["titulo"] + ". " + df["texto_limpo"]

print("✅ Limpeza concluída")
print("Total de artigos válidos:", len(df))
display(df[["titulo", "url"]])

✅ Limpeza concluída
Total de artigos válidos: 36


,titulo,url
0,TEA na educação infantil: inclusão e afetivida...,https://diversa.org.br/artigos/tea-educacao-in...
1,Ferramentas para o desenvolvimento escolar de ...,https://diversa.org.br/artigos/ferramentas-par...
2,O que é Transtorno do Déficit de Atenção e Hip...,https://diversa.org.br/artigos/o-que-e-transto...
3,Como a escola pode lidar com os diagnósticos m...,https://diversa.org.br/artigos/como-a-escola-p...
4,Psicomotricidade como recurso para a educação ...,https://diversa.org.br/artigos/psicomotricidad...
5,Braile: acessibilidade para estudantes com def...,https://diversa.org.br/noticias/braile-acessib...
6,Educação de surdos: os desafios da proposta bi...,https://diversa.org.br/artigos/educacao-de-sur...
7,Qual o papel do Tradutor/Intérprete de Libras ...,https://diversa.org.br/artigos/qual-o-papel-do...
8,Flexibilizações vs. adaptações curriculares: c...,https://diversa.org.br/artigos/flexibilizacoes...
9,Os desafios de ser uma mulher com deficiência ...,https://diversa.org.br/artigos/os-desafios-de-...


## 5. Vetorização TF-IDF e busca semântica


In [16]:
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=8000,
    sublinear_tf=True,
    strip_accents="unicode",
    lowercase=True,
)

matriz_tfidf = vectorizer.fit_transform(df["conteudo_busca"])

def buscar_artigos_relevantes(pergunta, top_k=3, score_minimo=0.015):
    vetor = vectorizer.transform([pergunta])
    scores = cosine_similarity(vetor, matriz_tfidf)[0]
    indices = scores.argsort()[::-1][:top_k]

    resultados = []
    for i in indices:
        score = float(scores[i])
        if score >= score_minimo:
            resultados.append({
                "titulo": df.iloc[i]["titulo"],
                "url": df.iloc[i]["url"],
                "trecho": df.iloc[i]["texto_limpo"][:900],
                "score": round(score, 4),
            })
    return resultados

print("✅ Vetorização concluída")
print("Matriz:", matriz_tfidf.shape)

✅ Vetorização concluída
Matriz: (36, 2540)


## 6. Teste da busca

In [17]:
pergunta_teste = "O que é Atendimento Educacional Especializado?"
resultados = buscar_artigos_relevantes(pergunta_teste)

print("Pergunta:", pergunta_teste)
print("Artigos recuperados:")
for r in resultados:
        print(f"- {r['titulo']} | score={r['score']}")

Pergunta: O que é Atendimento Educacional Especializado?
Artigos recuperados:
- O que é Atendimento Educacional Especializado (AEE)? | score=0.3487
- Atendimento educacional especializado (AEE): pressupostos e desafios | score=0.2507
- O que é a Política Nacional de Educação Especial na Perspectiva da Educação Inclusiva? | score=0.1147


## 7. Prompt do assistente e controle de escopo


In [18]:
# ── System Prompt ─────────────────────────────────────────────────────────────
SYSTEM_PROMPT_PROFESSOR = (
    "Você é um assistente especialista em Educação Inclusiva do Portal Diversa (diversa.org.br), "
    "respondendo a um PROFESSOR da Educação Básica ou do AEE.\n\n"
    "Seu papel é oferecer orientações pedagógicas detalhadas e aplicáveis em sala de aula, "
    "com base nos conteúdos do Portal Diversa.\n\n"
    "ESTILO DE RESPOSTA:\n"
    "- Use linguagem técnica-pedagógica, mas didática\n"
    "- Estruture a resposta em passos práticos sempre que possível\n"
    "- Explique o 'porquê' pedagógico das recomendações, não só o 'o quê'\n"
    "- Pode usar termos técnicos (AEE, DUA, adaptação curricular), mas explique-os brevemente\n\n"
    "RESPONDA perguntas sobre:\n"
    "- Deficiências e transtornos: autismo (TEA), TDAH, deficiência visual, auditiva, intelectual e física\n"
    "- Legislação: LBI, LDB, BNCC, Política Nacional de Educação Especial\n"
    "- Estratégias pedagógicas inclusivas e adaptações curriculares\n"
    "- Atendimento Educacional Especializado (AEE)\n"
    "- Tecnologia assistiva\n\n"
    "NAO RESPONDA (redirecione educadamente):\n"
    "- Diagnósticos médicos ou psicológicos\n"
    "- Opiniões políticas ou partidárias\n"
    "- Assuntos sem relação com educação inclusiva\n\n"
    "Se a pergunta estiver fora do escopo, responda exatamente:\n"
    "'Minha especialidade é Educação Inclusiva. Posso te ajudar com alguma dúvida sobre esse tema?'\n\n"
    "Responda sempre em português brasileiro. Cite o título e a URL do artigo consultado ao final."
)

SYSTEM_PROMPT_FAMILIA = (
    "Você é um assistente especialista em Educação Inclusiva do Portal Diversa (diversa.org.br), "
    "respondendo a um FAMILIAR de uma pessoa com deficiência.\n\n"
    "Seu papel é acolher, esclarecer dúvidas e orientar sobre direitos e recursos disponíveis, "
    "com base nos conteúdos do Portal Diversa.\n\n"
    "ESTILO DE RESPOSTA:\n"
    "- Use linguagem simples, calorosa e acessível, evitando jargão técnico\n"
    "- Quando precisar usar um termo técnico (ex: AEE, LBI), explique-o em uma frase simples\n"
    "- Demonstre empatia genuína, especialmente em temas sensíveis (diagnóstico, bullying, direitos)\n"
    "- Seja direto e prático: a família quer saber o que fazer, não uma aula teórica\n\n"
    "RESPONDA perguntas sobre:\n"
    "- Deficiências e transtornos: autismo (TEA), TDAH, deficiência visual, auditiva, intelectual e física\n"
    "- Direitos garantidos por lei (LBI, PNEEI)\n"
    "- Como a escola deve apoiar a família e o estudante\n"
    "- Tecnologia assistiva e recursos disponíveis\n\n"
    "NAO RESPONDA (redirecione educadamente):\n"
    "- Diagnósticos médicos ou psicológicos — sempre recomende um profissional habilitado\n"
    "- Opiniões políticas ou partidárias\n"
    "- Assuntos sem relação com educação inclusiva\n\n"
    "Se a pergunta estiver fora do escopo, responda exatamente:\n"
    "'Minha especialidade é Educação Inclusiva. Posso te ajudar com alguma dúvida sobre esse tema?'\n\n"
    "Responda sempre em português brasileiro, de forma acolhedora. Cite a fonte ao final, "
    "de forma simples (ex: 'Você pode ler mais sobre isso aqui: [link]')."
)

SYSTEM_PROMPT_GESTOR = (
    "Você é um assistente especialista em Educação Inclusiva do Portal Diversa (diversa.org.br), "
    "respondendo a um GESTOR escolar ou profissional de secretaria de educação.\n\n"
    "Seu papel é fornecer informações objetivas sobre legislação, políticas e gestão de recursos "
    "para inclusão, com base nos conteúdos do Portal Diversa.\n\n"
    "ESTILO DE RESPOSTA:\n"
    "- Use linguagem técnica, objetiva e direta\n"
    "- Priorize dados concretos: nomes de leis, decretos, percentuais, prazos quando disponíveis\n"
    "- Evite explicações longas de conceitos básicos — assuma conhecimento prévio do gestor\n"
    "- Estruture em tópicos quando a resposta envolver múltiplos itens (normas, responsabilidades)\n\n"
    "RESPONDA perguntas sobre:\n"
    "- Legislação: LBI, LDB, BNCC, Decreto 12.686/2025 (PNEEI), PNEEPEI\n"
    "- Gestão escolar inclusiva: projeto político-pedagógico, formação de professores\n"
    "- Atendimento Educacional Especializado (AEE) e Salas de Recursos Multifuncionais\n"
    "- Políticas públicas e estrutura de rede de ensino\n\n"
    "NAO RESPONDA (redirecione educadamente):\n"
    "- Diagnósticos médicos ou psicológicos\n"
    "- Decisões judiciais específicas de casos individuais — oriente buscar assessoria jurídica\n"
    "- Opiniões políticas ou partidárias\n\n"
    "Se a pergunta estiver fora do escopo, responda exatamente:\n"
    "'Minha especialidade é Educação Inclusiva. Posso te ajudar com alguma dúvida sobre esse tema?'\n\n"
    "Responda sempre em português brasileiro, de forma objetiva. Cite a fonte legal/documental ao final."
)

SYSTEM_PROMPT = SYSTEM_PROMPT_PROFESSOR  # compatibilidade com código antigo

# ============================================================
# VALIDAÇÃO DE ESCOPO
# ============================================================
# Corrige o erro: NameError: name 'pergunta_dentro_escopo' is not defined
# Essa função identifica se a pergunta pertence ao tema do assistente.

PALAVRAS_ESCOPO = [
    "educação", "inclus", "inclusiva", "deficiência", "deficiente", "autismo", "tea", "tdah",
    "aee", "lbi", "bncc", "ldb", "pnee", "pneepi", "aluno", "estudante", "criança",
    "escola", "professor", "docente", "gestor", "família", "responsável", "sala de recursos",
    "tecnologia assistiva", "acessibilidade", "libras", "braille", "curricular", "pedagóg",
    "aprendizagem", "apoio escolar", "laudo", "diagnóstico", "barreira", "adaptação",
    "adaptações", "desenho universal", "dua", "atendimento educacional especializado",
    "necessidades educacionais", "bullying", "mediação", "mediador", "intérprete", "recurso multifuncional"
]


def pergunta_dentro_escopo(pergunta):
    """
    Retorna True quando a pergunta parece estar relacionada à Educação Inclusiva.
    A função é usada para evitar respostas fora do tema do TCC.
    """
    pergunta_normalizada = limpar_texto(pergunta).lower()
    return any(palavra in pergunta_normalizada for palavra in PALAVRAS_ESCOPO)


print("✅ System prompt configurado!")

✅ System prompt configurado!


In [19]:
# ============================================================
# CONEXÃO ENTRE PROMPTS E DROPDOWN DE PERFIL
# ============================================================
# Esta célula faz a ponte entre o item 7 (prompts/escopo)
# e o item 10 (dropdown da interface Gradio).
#
# O valor escolhido no dropdown precisa ser exatamente uma das chaves abaixo:
# "Professor", "Família" ou "Gestor".

PROMPTS_POR_PERFIL = {
    "Professor": SYSTEM_PROMPT_PROFESSOR,
    "Família": SYSTEM_PROMPT_FAMILIA,
    "Gestor": SYSTEM_PROMPT_GESTOR,
}

# Aceita também variações sem acento/minúsculas, caso alguma parte antiga
# do notebook envie "familia", "professor" ou "gestor".
ALIASES_PERFIL = {
    "professor": "Professor",
    "familia": "Família",
    "família": "Família",
    "gestor": "Gestor",
    "Professor": "Professor",
    "Família": "Família",
    "Gestor": "Gestor",
}


def normalizar_perfil(perfil_usuario="Professor"):
    """
    Normaliza o perfil vindo do dropdown.
    Se vier vazio ou inválido, usa Professor como padrão seguro.
    """
    perfil_texto = str(perfil_usuario or "Professor").strip()
    return ALIASES_PERFIL.get(perfil_texto, "Professor")


def obter_system_prompt_por_perfil(perfil_usuario="Professor"):
    """
    Retorna o system prompt correto de acordo com o perfil selecionado.
    Essa função é chamada pela função responder() na Seção 8.
    """
    perfil_normalizado = normalizar_perfil(perfil_usuario)
    return PROMPTS_POR_PERFIL[perfil_normalizado]


print("✅ Prompts conectados ao dropdown de perfil!")


✅ Prompts conectados ao dropdown de perfil!


## 8. Funções de resposta



In [20]:
# ============================================================
# PERFIS DE USUÁRIO + RESPOSTA CONECTADA AO DROPDOWN
# ============================================================
# Esta seção usa o perfil escolhido no dropdown da interface Gradio.
# Fluxo final:
# Dropdown do item 10 → enviar_mensagem() → responder() → prompt correto do item 7.

PERFIS_USUARIO = {
    "Professor": {"temperature": 0.4, "max_tokens": 600, "frequency_penalty": 0.3},
    "Família":   {"temperature": 0.5, "max_tokens": 350, "frequency_penalty": 0.2},
    "Gestor":    {"temperature": 0.1, "max_tokens": 300, "frequency_penalty": 0.0},
}

INSTRUCOES_PERFIL = {
    "Professor": (
        "Perfil selecionado: Professor. Responda de forma pedagógica, detalhada e aplicável em sala de aula. "
        "Explique conceitos, traga orientações práticas e organize a resposta com clareza."
    ),
    "Família": (
        "Perfil selecionado: Família. Use linguagem simples, acolhedora e sem termos técnicos desnecessários. "
        "Explique com cuidado, empatia e foco em orientação prática para responsáveis e familiares."
    ),
    "Gestor": (
        "Perfil selecionado: Gestor. Responda de forma técnica, objetiva e estratégica. "
        "Priorize ações, responsabilidades, organização escolar, fluxos, políticas e tomada de decisão."
    ),
}

# Segurança para o caso de alguém executar esta célula sem executar a célula anterior.
if "PROMPTS_POR_PERFIL" not in globals():
    PROMPTS_POR_PERFIL = {
        "Professor": SYSTEM_PROMPT_PROFESSOR,
        "Família": SYSTEM_PROMPT_FAMILIA,
        "Gestor": SYSTEM_PROMPT_GESTOR,
    }

if "ALIASES_PERFIL" not in globals():
    ALIASES_PERFIL = {
        "professor": "Professor",
        "familia": "Família",
        "família": "Família",
        "gestor": "Gestor",
        "Professor": "Professor",
        "Família": "Família",
        "Gestor": "Gestor",
    }


def normalizar_perfil(perfil_usuario="Professor"):
    """Garante que sempre exista um perfil válido vindo do dropdown."""
    perfil_texto = str(perfil_usuario or "Professor").strip()
    return ALIASES_PERFIL.get(perfil_texto, "Professor")


def obter_system_prompt_por_perfil(perfil_usuario="Professor"):
    """Busca o system prompt correto para Professor, Família ou Gestor."""
    perfil_normalizado = normalizar_perfil(perfil_usuario)
    return PROMPTS_POR_PERFIL[perfil_normalizado]


def preparar_historico_para_modelo(historico=None, limite_mensagens=6):
    """
    Converte o histórico do Gradio para o formato aceito pela API.
    Usa apenas as últimas mensagens para manter a resposta contextual sem pesar o prompt.
    """
    if not historico:
        return []

    mensagens = []

    for item in historico[-limite_mensagens:]:
        if isinstance(item, dict):
            role = item.get("role", "assistant")
            content = item.get("content", "")
            if role in ("user", "assistant") and content:
                mensagens.append({"role": role, "content": str(content)})

        elif isinstance(item, (tuple, list)) and len(item) == 2:
            pergunta_antiga, resposta_antiga = item
            if pergunta_antiga:
                mensagens.append({"role": "user", "content": str(pergunta_antiga)})
            if resposta_antiga:
                mensagens.append({"role": "assistant", "content": str(resposta_antiga)})

    return mensagens


def chamar_groq(messages, perfil_usuario="Professor", tentativas=3):
    if not GROQ_KEY:
        return ""

    perfil_usuario = normalizar_perfil(perfil_usuario)
    parametros_perfil = PERFIS_USUARIO[perfil_usuario]

    headers = {
        "Authorization": f"Bearer {GROQ_KEY}",
        "Content-Type": "application/json",
    }

    # Os parâmetros do perfil sobrescrevem os parâmetros gerais.
    payload = {
        **PARAMS_GERACAO,
        **parametros_perfil,
        "messages": messages,
    }

    for tentativa in range(1, tentativas + 1):
        try:
            resp = requests.post(GROQ_URL, headers=headers, json=payload, timeout=30)

            if resp.status_code == 200:
                try:
                    dados = resp.json()
                    return dados["choices"][0]["message"]["content"].strip()
                except Exception as exc:
                    return (
                        "❌ A Groq respondeu, mas o retorno não veio no formato esperado. "
                        f"Detalhe técnico: {exc}"
                    )

            if resp.status_code == 429:
                time.sleep(5 * tentativa)
                continue

            if resp.status_code == 401:
                return "❌ Chave da Groq inválida. Verifique a variável GROQ_KEY."

            return f"❌ Erro HTTP {resp.status_code}: {resp.text[:250]}"

        except Exception as exc:
            return f"❌ Erro inesperado ao chamar a Groq: {exc}"

    return "⏳ Serviço indisponível no momento."


def resposta_demo_sem_llm(pergunta, artigos, perfil_usuario="Professor"):
    perfil_usuario = normalizar_perfil(perfil_usuario)

    if not artigos:
        return (
            "Minha especialidade é Educação Inclusiva. A base atual ainda não possui trechos suficientes "
            "para responder essa pergunta com segurança. Recomendo ampliar a base com mais artigos do Portal Diversa."
        )

    principal = artigos[0]

    if perfil_usuario == "Família":
        introducao = (
            "**Resposta em modo demonstração, sem API da Groq configurada:**\n\n"
            f"Pelo artigo **{principal['titulo']}**, a ideia principal é acolher o estudante e observar quais barreiras "
            "podem estar dificultando sua participação na escola. A família pode conversar com a equipe escolar, "
            "compartilhar informações importantes e acompanhar os apoios combinados.\n\n"
        )
    elif perfil_usuario == "Gestor":
        introducao = (
            "**Resposta em modo demonstração, sem API da Groq configurada:**\n\n"
            f"Com base no artigo **{principal['titulo']}**, a gestão deve mapear barreiras, organizar apoios pedagógicos, "
            "articular professores e AEE, acompanhar registros e garantir acessibilidade nas práticas escolares.\n\n"
        )
    else:
        introducao = (
            "**Resposta em modo demonstração, sem API da Groq configurada:**\n\n"
            f"Com base no artigo **{principal['titulo']}**, a orientação principal é considerar a educação inclusiva "
            "como um processo de remoção de barreiras para que todos os estudantes participem da vida escolar. "
            "A escola deve planejar estratégias pedagógicas acessíveis, articular o Atendimento Educacional Especializado "
            "quando necessário e envolver professores, gestão, família e comunidade.\n\n"
        )

    return (
        introducao
        + "Trecho recuperado da base:\n"
        + f"> {principal['trecho'][:550]}...\n\n"
        + "Para respostas mais completas geradas por IA, configure a variável `GROQ_KEY`."
    )


def montar_fontes(artigos):
    if not artigos:
        return ""
    linhas = ["\n\n---", "📚 **Fontes recuperadas da base:**"]
    for artigo in artigos:
        linhas.append(f"- [{artigo['titulo']}]({artigo['url']}) — score {artigo['score']}")
    return "\n".join(linhas)


# ============================================================
# CORREÇÃO DEFENSIVA DE ESCOPO
# ============================================================
# Mantém a célula funcionando mesmo se a célula anterior não tiver sido executada.

if "PALAVRAS_ESCOPO" not in globals():
    PALAVRAS_ESCOPO = [
        "educação", "inclus", "inclusiva", "deficiência", "deficiente", "autismo", "tea", "tdah",
        "aee", "lbi", "bncc", "ldb", "pnee", "pneepi", "aluno", "estudante", "criança", "escola",
        "professor", "docente", "gestor", "família", "responsável", "sala de recursos", "tecnologia assistiva",
        "acessibilidade", "libras", "braille", "curricular", "pedagóg", "aprendizagem", "apoio escolar",
        "laudo", "diagnóstico", "barreira", "adaptação", "adaptações", "desenho universal", "dua",
        "atendimento educacional especializado", "necessidades educacionais", "bullying", "mediação", "mediador",
        "intérprete", "recurso multifuncional"
    ]


def pergunta_dentro_escopo(pergunta):
    pergunta_normalizada = limpar_texto(pergunta).lower()
    return any(palavra in pergunta_normalizada for palavra in PALAVRAS_ESCOPO)


def responder(pergunta, historico=None, perfil_usuario="Professor"):
    """
    Pipeline RAG completo:
    1. Recebe a pergunta e o perfil escolhido no dropdown.
    2. Seleciona o system prompt correto do item 7.
    3. Usa somente o histórico interno do perfil ativo atual.
    4. Busca fontes na base.
    5. Envia tudo para a Groq ou usa resposta demo quando não houver chave.
    """
    pergunta = (pergunta or "").strip()
    perfil_usuario = normalizar_perfil(perfil_usuario)

    if not pergunta:
        return "Digite uma pergunta sobre Educação Inclusiva."

    artigos = buscar_artigos_relevantes(pergunta, top_k=3)

    if not pergunta_dentro_escopo(pergunta) and not artigos:
        return "Minha especialidade é Educação Inclusiva. Posso te ajudar com alguma dúvida sobre esse tema?"

    contexto = "\n\n".join(
        f"Artigo: {a['titulo']}\nURL: {a['url']}\nTrecho: {a['trecho']}\nScore: {a['score']}"
        for a in artigos
    ) or "Nenhum trecho relevante foi encontrado na base atual."

    instrucao_usuario = (
        "Use os TRECHOS DO PORTAL DIVERSA abaixo para responder. "
        "Não invente fontes, títulos ou URLs. Caso a informação não esteja nos trechos, "
        "diga que a base atual é insuficiente.\n\n"
        f"TRECHOS DO PORTAL DIVERSA:\n{contexto}\n\n"
        f"PERGUNTA DO USUÁRIO: {pergunta}"
    )

    system_prompt_escolhido = obter_system_prompt_por_perfil(perfil_usuario)

    messages = [
        {
            "role": "system",
            "content": f"{system_prompt_escolhido}\n\n{INSTRUCOES_PERFIL[perfil_usuario]}",
        },
        *preparar_historico_para_modelo(historico),
        {"role": "user", "content": instrucao_usuario},
    ]

    resposta = chamar_groq(messages, perfil_usuario=perfil_usuario)
    if not resposta:
        resposta = resposta_demo_sem_llm(pergunta, artigos, perfil_usuario=perfil_usuario)

    return f"**Perfil selecionado:** {perfil_usuario}\n\n" + resposta


print("✅ Funções criadas com dropdown conectado aos prompts de Professor, Família e Gestor")


✅ Funções criadas com dropdown conectado aos prompts de Professor, Família e Gestor


## 9. Testes automáticos do protótipo




In [21]:
# ════════════════════════════════════════════════════════════════════════════
# Seção 9 — Bateria de Testes do Assistente
# Categorias: dentro do escopo, fora do escopo, sensível, caso-limite
# Observação: execute as seções anteriores antes desta bateria.
# ════════════════════════════════════════════════════════════════════════════

testes = [
    # ✅ Dentro do escopo
    ("Dentro do escopo", "O que é tecnologia assistiva?"),
    ("Dentro do escopo", "Como trabalhar com alunos com TDAH em sala?"),
    ("Dentro do escopo", "Quais são os direitos do aluno com deficiência?"),
    ("Dentro do escopo", "O que diz o Decreto 12.686/2025 sobre AEE?"),
    ("Dentro do escopo", "Como adaptar atividades para alunos com deficiência visual?"),
    ("Dentro do escopo", "O que é uma sala de recursos multifuncionais?"),
    ("Dentro do escopo", "Como orientar a família de uma criança autista recém-diagnosticada?"),

    # 🚫 Fora do escopo
    ("Fora do escopo", "Qual é a capital da França?"),
    ("Fora do escopo", "Me dê uma receita de bolo."),
    ("Fora do escopo", "Quem você acha que vai ganhar a próxima eleição?"),

    # ⚠️ Sensível (dentro do escopo, mas exige cuidado)
    ("Sensível", "Minha filha tem laudo de autismo, ela tem direito a quê na escola?"),
    ("Sensível", "É melhor escola especial ou escola inclusiva?"),
    ("Sensível", "Minha filha está sofrendo bullying por ter deficiência, o que eu faço?"),

    # 🔶 Caso-limite (ambíguo)
    ("Caso-limite", "Como ajudar uma criança?"),
    ("Caso-limite", "Meu filho tem dificuldade de aprender, o que eu faço?"),
]

# ── Executa os testes e monta o relatório ──────────────────────────────────
resultados = []

print("🧪 Executando bateria de testes...")
print("=" * 70)

for categoria, pergunta in testes:
    print(f"\n[{categoria}] {pergunta}")
    resposta = responder(pergunta)
    print(f"→ {resposta[:250]}{'...' if len(resposta) > 250 else ''}")
    print("-" * 70)

    resultados.append({
        "categoria": categoria,
        "pergunta": pergunta,
        "resposta": resposta,
        "correta": "",      # <- vocês preenchem: SIM / NÃO
        "observacao": "",   # <- vocês preenchem: por que acertou/errou
    })

df_testes = pd.DataFrame(resultados)
print("\n✅ Testes concluídos! Tabela de avaliação criada em df_testes.")
print("   Preencha as colunas 'correta' e 'observacao' avaliando cada resposta.")

🧪 Executando bateria de testes...

[Dentro do escopo] O que é tecnologia assistiva?
→ **Perfil selecionado:** Professor

A tecnologia assistiva é um recurso importante para garantir a acessibilidade e a inclusão de estudantes com deficiência ou necessidades especiais na educação. Segundo o Portal Diversa, a tecnologia assistiva é um c...
----------------------------------------------------------------------

[Dentro do escopo] Como trabalhar com alunos com TDAH em sala?
→ **Perfil selecionado:** Professor

Trabalhar com alunos com Transtorno do Deficit de Atenção e Hiperatividade (TDAH) em sala de aula requer uma abordagem pedagógica específica, que priorize a flexibilidade, a organização e a motivação. Aqui estão alg...
----------------------------------------------------------------------

[Dentro do escopo] Quais são os direitos do aluno com deficiência?
→ **Perfil selecionado:** Professor

Excelente pergunta!

De acordo com a legislação federal brasileira, os alunos com deficiência

## 10. Interface de chat com Gradio


In [ ]:
import base64
from pathlib import Path
from html import escape

import gradio as gr


# ============================================================
# CONFIGURAÇÕES DO PROJETO
# ============================================================

PASTA_IMAGENS = Path("./imagens_do_tcc")

CAMINHO_LOGO_ACCENTURE = PASTA_IMAGENS / "logo_accenture.png"
CAMINHO_CORDAO = PASTA_IMAGENS / "CordaoTEA.svg"
CAMINHO_ICONE_GRUPO = PASTA_IMAGENS / "images.svg"
CAMINHO_LOGO_DIVERSA = PASTA_IMAGENS / "logo_diversa.png"
CAMINHO_CSS = Path("style.css")


# ============================================================
# FUNÇÕES AUXILIARES
# ============================================================

def carregar_imagem_base64(caminho: Path, tipo_mime: str) -> str:
    """
    Carrega uma imagem local e retorna uma URL base64 para usar no HTML/CSS.
    Exemplo de retorno: data:image/png;base64,...
    """
    if not caminho.exists():
        raise FileNotFoundError(
            f"Imagem não encontrada: {caminho}\n"
            "Verifique se a pasta 'imagens_do_tcc' está no mesmo local deste arquivo."
        )

    with caminho.open("rb") as arquivo:
        imagem_base64 = base64.b64encode(arquivo.read()).decode("utf-8")

    return f"data:{tipo_mime};base64,{imagem_base64}"


def carregar_css() -> str:
    """
    Lê o arquivo style.css e troca os marcadores pelas imagens em base64.
    Isso deixa o Python mais limpo e o CSS mais fácil de manter.
    """
    if not CAMINHO_CSS.exists():
        raise FileNotFoundError(
            "Arquivo style.css não encontrado.\n"
            "Deixe o notebook/app.py e o style.css na mesma pasta."
        )

    css = CAMINHO_CSS.read_text(encoding="utf-8")

    substituicoes = {
        "__CORDAO_TEA_URL__": CORDAO_TEA_URL,
        "__LOGO_ACCENTURE_URL__": LOGO_ACCENTURE_URL,
    }

    for marcador, valor in substituicoes.items():
        css = css.replace(marcador, valor)

    return css


# ============================================================
# IMAGENS DO FRONT
# ============================================================

LOGO_ACCENTURE_URL = carregar_imagem_base64(CAMINHO_LOGO_ACCENTURE, "image/png")
CORDAO_TEA_URL = carregar_imagem_base64(CAMINHO_CORDAO, "image/svg+xml")
ICONE_GRUPO_URL = carregar_imagem_base64(CAMINHO_ICONE_GRUPO, "image/svg+xml")

try:
    LOGO_DIVERSA_URL = carregar_imagem_base64(CAMINHO_LOGO_DIVERSA, "image/png")
except FileNotFoundError:
    LOGO_DIVERSA_URL = None


def montar_bloco_parceria_diversa():
    """Monta o selo visual da DIVERSA no cabeçalho."""
    if LOGO_DIVERSA_URL:
        return f"""
        <div class="parceria-diversa">
            <span>Conteúdo parceiro</span>
            <img src="{LOGO_DIVERSA_URL}" alt="Logo DIVERSA">
        </div>
        """

    return """
    <div class="parceria-diversa">
        <span>Conteúdo parceiro</span>
        <strong>DIVERSA</strong>
    </div>
    """


# ============================================================
# CSS PERSONALIZADO
# ============================================================

css_personalizado = carregar_css()

# CSS complementar: card lateral de fontes consultadas.
# Mantém a essência do layout antigo e adiciona apenas o painel visual de fontes.
CSS_FONTES_CARD = """
/* ============================================================
   CARD DE FONTES CONSULTADAS
   ============================================================ */

#fontes-painel {
    width: 100% !important;
}

#fontes-card {
    width: 100% !important;
    padding: 20px 18px !important;

    background: #FFFFFF !important;
    border: 1.5px solid rgba(161, 0, 255, 0.35) !important;
    border-radius: 22px !important;

    box-shadow: none !important;
}

.fontes-titulo {
    margin: 0 0 14px 0 !important;
    color: #111827 !important;
    font-size: 16px !important;
    font-weight: 800 !important;
    letter-spacing: -0.15px !important;
}

.fonte-vazia {
    margin: 0 !important;
    color: #6B7280 !important;
    font-size: 13px !important;
    line-height: 1.5 !important;
}

.fonte-item {
    padding: 12px 0 !important;
    border-top: 1px solid rgba(17, 24, 39, 0.10) !important;
}

.fonte-item:first-of-type {
    border-top: 0 !important;
    padding-top: 2px !important;
}

.fonte-item a {
    display: block !important;
    color: #111827 !important;
    font-size: 14px !important;
    font-weight: 800 !important;
    line-height: 1.4 !important;
    text-decoration: none !important;
}

.fonte-item a:hover {
    color: #6F00B8 !important;
    text-decoration: underline !important;
}

.fonte-score {
    display: block !important;
    margin-top: 5px !important;
    color: #374151 !important;
    font-size: 13px !important;
    line-height: 1.4 !important;
}
"""

css_personalizado = css_personalizado + "\n" + CSS_FONTES_CARD



# ============================================================
# FUNÇÕES DO CHAT
# ============================================================

PERFIS_CHAT = ["Professor", "Família", "Gestor"]


def criar_historico_inicial(perfil_usuario="Professor"):
    """Cria a primeira mensagem visual de cada perfil."""
    perfil_usuario = normalizar_perfil(perfil_usuario)
    return [
        {
            "role": "assistant",
            "content": (
                f"Olá! Sou o Assistente Diversa em modo **{perfil_usuario}**. "
                "Como posso ajudar com educação inclusiva?"
            )
        }
    ]


def criar_historicos_visuais_iniciais():
    """Cria um histórico visual separado para cada tipo de usuário."""
    return {perfil: criar_historico_inicial(perfil) for perfil in PERFIS_CHAT}


def criar_historicos_modelo_iniciais():
    """Cria um histórico interno separado para cada tipo de usuário."""
    return {perfil: [] for perfil in PERFIS_CHAT}


def criar_fontes_por_perfil_iniciais():
    """Cria o estado inicial do painel de fontes para cada perfil."""
    return {perfil: montar_fontes_painel() for perfil in PERFIS_CHAT}


MENSAGEM_INICIAL_CHAT = criar_historico_inicial("Professor")

# CORREÇÃO PRINCIPAL:
# O erro "Token inesperado '<'" acontecia porque o backend estava devolvendo
# histórico no formato antigo de tuplas, mas a versão atual do Gradio espera
# mensagens no formato:
# [{"role": "user", "content": "..."}, {"role": "assistant", "content": "..."}]
#
# Por isso, abaixo o Chatbot trabalha sempre no formato "messages".


def criar_chatbot_compativel():
    """
    Cria o Chatbot usando o formato de mensagens.
    Funciona em versões recentes do Gradio.
    """
    parametros = {
        "label": "",
        "show_label": False,
        "elem_id": "chatbot",
        "height": 490,
        "value": MENSAGEM_INICIAL_CHAT.copy(),
    }

    try:
        # Gradio 4/5
        return gr.Chatbot(**parametros, type="messages")
    except TypeError:
        # Gradio 6+
        return gr.Chatbot(**parametros)



def extrair_texto_conteudo_chat(conteudo):
    """
    Algumas versões do Gradio devolvem o conteúdo da mensagem como texto,
    e outras devolvem como lista de partes: [{"text": "...", "type": "text"}].
    Esta função normaliza tudo para texto puro.
    """
    if conteudo is None:
        return ""

    if isinstance(conteudo, str):
        return conteudo

    if isinstance(conteudo, list):
        textos = []
        for parte in conteudo:
            if isinstance(parte, dict):
                texto = parte.get("text") or parte.get("content") or ""
                if texto:
                    textos.append(str(texto))
            else:
                textos.append(str(parte))
        return "\n".join(textos)

    if isinstance(conteudo, dict):
        return str(conteudo.get("text") or conteudo.get("content") or "")

    return str(conteudo)


def normalizar_historico_chat(historico):
    """
    Garante que o histórico sempre fique no formato messages.
    """
    historico_messages = []

    if not historico:
        return historico_messages

    for item in historico:
        if isinstance(item, dict) and "role" in item and "content" in item:
            role = item.get("role", "assistant")
            if role not in ("user", "assistant"):
                role = "assistant"

            historico_messages.append({
                "role": role,
                "content": extrair_texto_conteudo_chat(item.get("content", ""))
            })

        elif isinstance(item, (tuple, list)) and len(item) == 2:
            pergunta_antiga, resposta_antiga = item

            if pergunta_antiga:
                historico_messages.append({
                    "role": "user",
                    "content": str(pergunta_antiga)
                })

            if resposta_antiga:
                historico_messages.append({
                    "role": "assistant",
                    "content": str(resposta_antiga)
                })

    return historico_messages



def montar_fontes_painel_por_artigos(artigos):
    """
    Monta o card visual de fontes consultadas.
    Esse card aparece separado do chatbot, deixando a interface mais profissional.
    """
    if not artigos:
        return """
        <div id="fontes-card">
            <h3 class="fontes-titulo">Fontes consultadas</h3>
            <p class="fonte-vazia">
                As fontes mais relevantes aparecerão aqui depois da primeira pergunta.
            </p>
        </div>
        """

    itens = []

    for indice, artigo in enumerate(artigos, start=1):
        titulo = escape(str(artigo.get("titulo", "Fonte sem título")))
        url = escape(str(artigo.get("url", "#") or "#"))
        score = escape(str(artigo.get("score", "")))

        itens.append(f"""
        <div class="fonte-item">
            <a href="{url}" target="_blank" rel="noopener noreferrer">
                [{indice}] {titulo}
            </a>
            <span class="fonte-score">Relevância na busca: {score}</span>
        </div>
        """)

    return f"""
    <div id="fontes-card">
        <h3 class="fontes-titulo">Fontes consultadas</h3>
        {''.join(itens)}
    </div>
    """


def montar_fontes_painel(pergunta=""):
    """
    Busca os artigos relevantes da pergunta e monta o card lateral de fontes.
    """
    pergunta = (pergunta or "").strip()

    if not pergunta:
        return montar_fontes_painel_por_artigos([])

    try:
        artigos = buscar_artigos_relevantes(pergunta, top_k=3)
    except Exception:
        artigos = []

    return montar_fontes_painel_por_artigos(artigos)


def garantir_historicos_visuais_por_perfil(historicos_visuais_por_perfil=None):
    """
    Garante que exista um histórico visual separado para Professor, Família e Gestor.
    Esse histórico é o que aparece na tela do Chatbot.
    """
    historicos = criar_historicos_visuais_iniciais()

    if isinstance(historicos_visuais_por_perfil, dict):
        for perfil, historico in historicos_visuais_por_perfil.items():
            perfil_normalizado = normalizar_perfil(perfil)
            historico_normalizado = normalizar_historico_chat(historico)
            historicos[perfil_normalizado] = historico_normalizado or criar_historico_inicial(perfil_normalizado)

    return historicos


def garantir_historicos_modelo_por_perfil(historicos_modelo_por_perfil=None):
    """
    Garante que exista um histórico interno separado para Professor, Família e Gestor.
    Esse histórico é enviado para a IA e evita misturar contextos entre perfis.
    """
    historicos = criar_historicos_modelo_iniciais()

    if isinstance(historicos_modelo_por_perfil, dict):
        for perfil, historico in historicos_modelo_por_perfil.items():
            perfil_normalizado = normalizar_perfil(perfil)
            historicos[perfil_normalizado] = normalizar_historico_chat(historico)

    return historicos


def garantir_fontes_por_perfil(fontes_por_perfil=None):
    """Garante um painel de fontes separado para cada perfil."""
    fontes = criar_fontes_por_perfil_iniciais()

    if isinstance(fontes_por_perfil, dict):
        for perfil, html_fontes in fontes_por_perfil.items():
            perfil_normalizado = normalizar_perfil(perfil)
            fontes[perfil_normalizado] = html_fontes or montar_fontes_painel()

    return fontes


def trocar_perfil(
    novo_perfil,
    historico_visual_atual,
    perfil_ativo,
    historicos_visuais_por_perfil,
    historicos_modelo_por_perfil,
    fontes_por_perfil,
):
    """
    Salva a conversa do perfil atual e carrega a conversa do novo perfil.

    Exemplo:
    - Se você estava em Gestor, a conversa de Gestor fica guardada.
    - Ao mudar para Professor, a tela mostra a conversa de Professor.
    - Ao voltar para Gestor, a conversa de Gestor reaparece.
    """
    novo_perfil = normalizar_perfil(novo_perfil)
    perfil_ativo = normalizar_perfil(perfil_ativo)

    historicos_visuais = garantir_historicos_visuais_por_perfil(historicos_visuais_por_perfil)
    historicos_modelo = garantir_historicos_modelo_por_perfil(historicos_modelo_por_perfil)
    fontes = garantir_fontes_por_perfil(fontes_por_perfil)

    # Antes de sair do perfil atual, salva exatamente o que está na tela.
    historico_atual = normalizar_historico_chat(historico_visual_atual)
    historicos_visuais[perfil_ativo] = historico_atual or criar_historico_inicial(perfil_ativo)

    # Depois carrega o histórico visual e as fontes do perfil escolhido.
    historico_novo = historicos_visuais.get(novo_perfil) or criar_historico_inicial(novo_perfil)
    fontes_novo = fontes.get(novo_perfil) or montar_fontes_painel()

    return historico_novo, fontes_novo, novo_perfil, historicos_visuais, historicos_modelo, fontes


def enviar_mensagem(
    mensagem,
    historico_visual_atual,
    perfil_usuario="Professor",
    perfil_ativo="Professor",
    historicos_visuais_por_perfil=None,
    historicos_modelo_por_perfil=None,
    fontes_por_perfil=None,
):
    """
    Envia a pergunta usando histórico separado por perfil.

    Agora existem três conversas independentes:
    - Professor
    - Família
    - Gestor

    Ao trocar o dropdown, a conversa antiga some da tela, mas fica salva
    no perfil correspondente. Quando voltar para aquele perfil, ela reaparece.
    """
    perfil_usuario = normalizar_perfil(perfil_usuario)
    perfil_ativo = normalizar_perfil(perfil_ativo)

    historicos_visuais = garantir_historicos_visuais_por_perfil(historicos_visuais_por_perfil)
    historicos_modelo = garantir_historicos_modelo_por_perfil(historicos_modelo_por_perfil)
    fontes = garantir_fontes_por_perfil(fontes_por_perfil)

    historico_visual_atual = normalizar_historico_chat(historico_visual_atual)

    # Proteção extra: se o dropdown mudou e o evento change ainda não terminou,
    # salva o perfil antigo e carrega o histórico do novo perfil antes de responder.
    if perfil_usuario != perfil_ativo:
        historicos_visuais[perfil_ativo] = historico_visual_atual or criar_historico_inicial(perfil_ativo)
        historico_visual = historicos_visuais.get(perfil_usuario) or criar_historico_inicial(perfil_usuario)
        perfil_ativo = perfil_usuario
    else:
        historico_visual = historico_visual_atual or historicos_visuais.get(perfil_usuario) or criar_historico_inicial(perfil_usuario)

    historico_modelo = normalizar_historico_chat(historicos_modelo.get(perfil_usuario, []))

    if not mensagem or not mensagem.strip():
        fontes_atuais = fontes.get(perfil_usuario) or montar_fontes_painel()
        historicos_visuais[perfil_usuario] = historico_visual
        historicos_modelo[perfil_usuario] = historico_modelo
        return "", historico_visual, fontes_atuais, perfil_ativo, historicos_visuais, historicos_modelo, fontes

    mensagem = mensagem.strip()

    try:
        resposta = responder(mensagem, historico_modelo, perfil_usuario)
    except NameError:
        resposta = (
            "A função responder(mensagem) ainda não foi configurada. "
            "Conecte aqui o back-end/RAG do projeto para gerar a resposta."
        )
    except Exception as erro:
        resposta = (
            "Não consegui gerar a resposta neste momento.\n\n"
            f"Detalhe técnico: {type(erro).__name__}: {erro}"
        )

    nova_pergunta = {"role": "user", "content": mensagem}
    nova_resposta = {"role": "assistant", "content": str(resposta)}

    # Atualiza o histórico visual do perfil ativo.
    historico_visual.append(nova_pergunta)
    historico_visual.append(nova_resposta)
    historicos_visuais[perfil_usuario] = historico_visual

    # Atualiza o histórico interno do modelo somente para o perfil ativo.
    historico_modelo.append(nova_pergunta)
    historico_modelo.append(nova_resposta)
    historicos_modelo[perfil_usuario] = historico_modelo

    # Atualiza o painel de fontes somente para o perfil ativo.
    fontes_html = montar_fontes_painel(mensagem)
    fontes[perfil_usuario] = fontes_html

    return "", historico_visual, fontes_html, perfil_usuario, historicos_visuais, historicos_modelo, fontes


def limpar_chat(
    perfil_usuario="Professor",
    historicos_visuais_por_perfil=None,
    historicos_modelo_por_perfil=None,
    fontes_por_perfil=None,
):
    """
    Limpa somente a conversa do perfil selecionado.
    As conversas dos outros perfis continuam salvas.
    """
    perfil_usuario = normalizar_perfil(perfil_usuario)

    historicos_visuais = garantir_historicos_visuais_por_perfil(historicos_visuais_por_perfil)
    historicos_modelo = garantir_historicos_modelo_por_perfil(historicos_modelo_por_perfil)
    fontes = garantir_fontes_por_perfil(fontes_por_perfil)

    historico_limpo = criar_historico_inicial(perfil_usuario)
    fontes_limpas = montar_fontes_painel()

    historicos_visuais[perfil_usuario] = historico_limpo
    historicos_modelo[perfil_usuario] = []
    fontes[perfil_usuario] = fontes_limpas

    return historico_limpo, fontes_limpas, historicos_visuais, historicos_modelo, fontes


def preencher_pergunta(texto):
    """Preenche o campo de pergunta ao clicar em uma sugestão."""
    return texto


# ============================================================
# TEMA DO GRADIO
# ============================================================

tema = gr.themes.Soft(
    primary_hue="purple",
    neutral_hue="slate",
    font=gr.themes.GoogleFont("Inter")
)


# ============================================================
# INTERFACE
# ============================================================

with gr.Blocks(
    theme=tema,
    css=css_personalizado,
    title="Assistente Diversa — Educação Inclusiva"
) as demo:

    # Cabeçalho: título do grupo e dropdown na mesma linha.
    # O visual fica no style.css para manter o Python limpo.
    with gr.Row(elem_id="topo-hero"):
        with gr.Column(scale=5, elem_id="coluna-hero"):
            gr.HTML(f"""
            <div id="hero-conteudo">
                <h1 class="titulo-grupo">
                    <img src="{ICONE_GRUPO_URL}" class="icone-grupo" alt="Ícone do grupo">
                    <span>Acessibilidade Inteligente</span>
                    <span class="detalhe-roxo">&gt;</span>
                </h1>

                <p class="subtitulo-equipe">Equipe Hélice</p>
                <p class="descricao-hero">Assistente acadêmico para apoio à educação inclusiva, com respostas adaptadas ao perfil do usuário.</p>

                {montar_bloco_parceria_diversa()}
            </div>
            """)

        with gr.Column(scale=1, min_width=300, elem_id="box-perfil"):
            gr.HTML('<div class="perfil-label">Tipo de usuário</div>')

            # Dropdown conectado ao item 7:
            # a escolha altera o prompt, o estilo e os parâmetros da resposta.
            perfil_usuario = gr.Dropdown(
                choices=["Professor", "Família", "Gestor"],
                value="Professor",
                label=None,
                show_label=False,
                elem_id="perfil-rapido",
                interactive=True,
                allow_custom_value=False,
                filterable=False,
            )

            # Estados internos usados para histórico separado por perfil.
            # Cada perfil tem sua própria conversa visual e seu próprio contexto enviado à IA.
            perfil_ativo_estado = gr.State(value="Professor")
            historicos_visuais_por_perfil_estado = gr.State(value=criar_historicos_visuais_iniciais())
            historicos_modelo_por_perfil_estado = gr.State(value=criar_historicos_modelo_iniciais())
            fontes_por_perfil_estado = gr.State(value=criar_fontes_por_perfil_iniciais())

    with gr.Row(elem_id="linha-principal"):

        with gr.Column(scale=4):
            chatbot = criar_chatbot_compativel()

            pergunta = gr.Textbox(
                label="Digite sua pergunta",
                placeholder="Exemplo: O que é AEE? Como incluir um aluno com TEA?",
                lines=3,
                elem_id="pergunta-textbox"
            )

            with gr.Row(elem_id="linha-botoes"):
                btn_enviar = gr.Button(
                    "Enviar pergunta",
                    elem_id="btn-enviar",
                    elem_classes=["botao-principal"],
                    variant="secondary"
                )
                btn_limpar = gr.Button(
                    "Limpar histórico",
                    elem_id="btn-limpar",
                    elem_classes=["botao-principal"],
                    variant="secondary"
                )

        with gr.Column(scale=1, elem_id="coluna-direita"):

            fontes_painel = gr.HTML(
                value=montar_fontes_painel(),
                elem_id="fontes-painel"
            )

            gr.HTML("""
            <div class="card-info">
                <h3>Sobre o protótipo</h3>
                <p class="small-text">
                    Este assistente responde com base em uma base de conhecimento
                    cadastrada manualmente com fontes reais sobre educação inclusiva.
                </p>
                <p class="small-text">
                    Ele não substitui professores, profissionais do AEE, médicos,
                    psicólogos ou especialistas.
                </p>
            </div>
            """)

            with gr.Column(elem_id="card-sugestoes"):
                gr.HTML("""
                <h3>Perguntas sugeridas</h3>
                <p class="small-text">
                    Clique em uma pergunta para preencher o campo de texto.
                </p>
                """)

                with gr.Column(elem_id="lista-sugestoes"):
                    btn_p1 = gr.Button(
                        "O que é educação inclusiva?",
                        elem_classes=["pergunta-btn"]
                    )
                    btn_p2 = gr.Button(
                        "O que é AEE?",
                        elem_classes=["pergunta-btn"]
                    )
                    btn_p3 = gr.Button(
                        "Como incluir aluno com TEA?",
                        elem_classes=["pergunta-btn"]
                    )

    gr.HTML("""
    <div class="footer">
        Protótipo acadêmico para educação inclusiva — SoulCode / Accenture · Conteúdo de referência: DIVERSA
    </div>
    """)

    # ========================================================
    # EVENTOS DOS BOTÕES PRINCIPAIS
    # ========================================================

    perfil_usuario.change(
        fn=trocar_perfil,
        inputs=[
            perfil_usuario,
            chatbot,
            perfil_ativo_estado,
            historicos_visuais_por_perfil_estado,
            historicos_modelo_por_perfil_estado,
            fontes_por_perfil_estado,
        ],
        outputs=[
            chatbot,
            fontes_painel,
            perfil_ativo_estado,
            historicos_visuais_por_perfil_estado,
            historicos_modelo_por_perfil_estado,
            fontes_por_perfil_estado,
        ]
    )

    btn_enviar.click(
        fn=enviar_mensagem,
        inputs=[
            pergunta,
            chatbot,
            perfil_usuario,
            perfil_ativo_estado,
            historicos_visuais_por_perfil_estado,
            historicos_modelo_por_perfil_estado,
            fontes_por_perfil_estado,
        ],
        outputs=[
            pergunta,
            chatbot,
            fontes_painel,
            perfil_ativo_estado,
            historicos_visuais_por_perfil_estado,
            historicos_modelo_por_perfil_estado,
            fontes_por_perfil_estado,
        ]
    )

    pergunta.submit(
        fn=enviar_mensagem,
        inputs=[
            pergunta,
            chatbot,
            perfil_usuario,
            perfil_ativo_estado,
            historicos_visuais_por_perfil_estado,
            historicos_modelo_por_perfil_estado,
            fontes_por_perfil_estado,
        ],
        outputs=[
            pergunta,
            chatbot,
            fontes_painel,
            perfil_ativo_estado,
            historicos_visuais_por_perfil_estado,
            historicos_modelo_por_perfil_estado,
            fontes_por_perfil_estado,
        ]
    )

    btn_limpar.click(
        fn=limpar_chat,
        inputs=[
            perfil_usuario,
            historicos_visuais_por_perfil_estado,
            historicos_modelo_por_perfil_estado,
            fontes_por_perfil_estado,
        ],
        outputs=[
            chatbot,
            fontes_painel,
            historicos_visuais_por_perfil_estado,
            historicos_modelo_por_perfil_estado,
            fontes_por_perfil_estado,
        ]
    )

    # ========================================================
    # EVENTOS DAS PERGUNTAS SUGERIDAS
    # ========================================================

    btn_p1.click(
        fn=lambda: preencher_pergunta("O que é educação inclusiva?"),
        inputs=[],
        outputs=[pergunta]
    )

    btn_p2.click(
        fn=lambda: preencher_pergunta("O que é Atendimento Educacional Especializado?"),
        inputs=[],
        outputs=[pergunta]
    )

    btn_p3.click(
        fn=lambda: preencher_pergunta("Como incluir um aluno com TEA em sala de aula?"),
        inputs=[],
        outputs=[pergunta]
    )


# ============================================================
# EXECUÇÃO DO APP
# ============================================================

if __name__ == "__main__":
    demo.launch(
        share=True,
        inbrowser=True,
        debug=True
    )


C:\Users\Cesar\AppData\Local\Temp\ipykernel_13708\1768801315.py:575: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://7ba3b9812e6cb714e6.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


c:\Users\Cesar\anaconda3\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\Cesar\anaconda3\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\Cesar\anaconda3\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\Cesar\anaconda3\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\Cesar\a

# Respostas objetivas

## Requisito 1: JP (ok)

**1. Expandir a base de conhecimento**
  Copie manualmente pelo menos **20 artigos** do Portal Diversa cobrindo os principais
  temas do público-alvo: autismo (TEA), TDAH, deficiência visual, auditiva, intelectual,
  tecnologia assistiva, AEE, legislação (LBI, BNCC) e sala de recursos multifuncionais.
  Cada artigo deve ter `titulo`, `url` e `texto` com pelo menos 300 caracteres.

### Resposta


Nossa base de dados tem ao menos 2 artigos de cada tópico solicitado, e alguns temas específicos de finalidade da própria plataforma da Diversa.

Atualmente, o sistema conta com 20 artigos recolhidos e guardados no Google Sheets. O consumo é via link do sheet.

Para aumentar o alinhamento de palavras para o tema, também foram incluídos alguns outros artigos alternativos referentes aos temas:
- Funcionamento e finalidade da plataforma DIVERSA.
- Textos menores que trazem definições de termos mais gerais (ex: o que é educação inclusiva) com base em resumos de IA feito sobre os artigos para diversificar o vocabulário e trazer um tom mais didático.

### Expectativas futuras

- Scraping automático (ex. crawler com espaçamento temporal de requisições, pdfs, etc)
- Realizar um tratamento melhor do texto do artigo

## Requisito 2* - Nicolas


**2. Corrigir a alucinação. Por exemplo, para a de URLs**
  Adicionar ao `SYSTEM_PROMPT` a instrução: *"Nunca invente URLs ou títulos de artigos.
  Cite apenas fontes que estejam nos TRECHOS fornecidos."*
  Testar com perguntas sobre temas que não estão na base e confirmar que o modelo
  reconhece a limitação em vez de inventar fontes.

### Resposta

Esperando retorno de Nicolas.

## Requisito 3* - Nicolas



- [X] **3. Realizar e documentar os testes**
  Executar a bateria de testes da Seção 9 e documentar os resultados reais —
  quais perguntas foram respondidas corretamente, quais foram redirecionadas
  e quais ainda precisam de melhoria.

### Resposta

Esperando retorno de Nicolas.

## Requisito 4* - Nicolas


- [???] **4. Preencher a Seção de Ética (Seção 10)**
  Completar a análise com base nos testes realizados: quais limitações foram
  observadas na prática, como o sistema se comportou com perguntas fora do escopo
  e quais riscos existem no uso em ambiente escolar real.

## Requisito 5 - JP (ok)


- **5. Documentar a escolha do modelo**
  Justificar por que o `llama-3.1-8b-instant` (ou outro) foi escolhido, sua licença de uso,
  os limites do free tier e o que aconteceria se o limite diário fosse atingido.


### Resposta 

1. **Perfeito para Robôs Inteligentes (RAG Agêntico)**

    - Conecta com Ferramentas: O Llama 3.1 foi treinado de fábrica para decidir quando precisa usar ferramentas externas, como a sua busca de artigos.

    - Memória Gigante (128k): Ele consegue ler e cruzar dados de vários artigos longos ao mesmo tempo sem esquecer as informações do início do texto.

    - Referência: 
        - Documento oficial de lançamento: AI@Meta (2024) – The Llama 3 Herd of Models.
        - Estudo sobre modelos comerciais: Minaee, S., et al. (2024) – Large Language Models: A Survey.

2. **Rápido para Organizar e Formatar Textos**

    - Entrega Pronta: Ele é especialista em ler textos acadêmicos confusos e organizar tudo em formatos limpos (como tabelas ou JSON).

    - Super Veloz: Por ser um modelo leve (8B), ele processa volumes enormes de artigos em segundos, sem travar o sistema.

    - Estudo sobre modelos comerciais: Minaee, S., et al. (2024) – Large Language Models: A Survey.

3. **Segurança e Controle Total (Open Source)**
    - Dados Protegidos: Como o modelo é aberto, os seus artigos não são enviados para empresas terceiras. Tudo roda sob o seu controle.

    - Sem Dependência: Você não fica refém dos preços ou das regras de empresas como a OpenAI. O projeto é 100% seu.

    - Referência: Relatório de Stanford sobre IA: Bommasani, R., et al. (Stanford CRFM) – On the Opportunities and Risks of Foundation Models.

4. **Possibilidade de uso diário com Free tier**
    - Uso controlado gratuito com os seguintes limites: 
        - 30 requests/min
        - 14.4K requests/dia.
        - 6K tokens/min
        - 500K tokens/dia

    - Referência: https://console.groq.com/settings/limits

5. **Limite atingido**
    - Responderia com HTTP 429 (Too Many Requests)
    - É considerável ter um plano de Fallback para outra IA em caso de exceder os limites de requisição.

6. **Licença de Uso**
    - Comercialização: Gratuito para uso comercial, modificação e fine-tuning.

    - Teto de Usuários: Exige licença paga da Meta se o seu produto passar de 700 milhões de usuários ativos mensais.

    - Propriedade Intelectual: Proibido usar os dados gerados pelo Llama para treinar ou melhorar modelos concorrentes.

    - Vedações (AUP): Proibido uso militar, infraestrutura crítica, crimes virtuais e aconselhamento autônomo (médico/jurídico).

    - Créditos: Obrigatório incluir a marca "Built with Meta Llama" na sua aplicação.

    - Referências: 
        - Meta Llama License - (https://developer.meta.com/ai/llama3/license/)
        - Acceptable Use Policy - (https://developer.meta.com/ai/llama3/use-policy/)


## Requisito 6* - Brena


- [BRENDA] **6. Preparar os slides de apresentação**
  Ver seção **📊 Slides do TCC** mais abaixo nesta célula.


  -- Validar com ela

## Requisito 7 - Cesar (falta o texto)


**7. Personalizar a interface de chat**

- Desenvolvimento visual do front-end

    - Foi criado um CSS personalizado para o protótipo, com foco em deixar a interface mais profissional, organizada e adequada para uma apresentação acadêmica.

    - Foram definidas variáveis visuais globais, como cores principais, tons de roxo, cores de texto, sombras, bordas e arredondamentos. Isso ajudou a manter uma identidade visual consistente em toda a aplicação.

    - A interface recebeu uma identidade visual mais próxima da proposta do projeto, usando o roxo como cor de destaque, remetendo também à estética institucional da Accenture.

    - O layout principal foi ajustado para ocupar melhor a tela, com melhor espaçamento entre os blocos, colunas mais organizadas e uma distribuição visual mais equilibrada entre chatbot, cards laterais e área de perguntas.


- Melhorias no cabeçalho do sistema

    - Foi criado um cabeçalho mais profissional para o projeto, destacando o nome “Acessibilidade Inteligente”, a equipe e a proposta do assistente.

    - Foi adicionada uma área visual para reforçar o vínculo com o conteúdo do DIVERSA, usando um selo de parceria institucional.

    - O cabeçalho recebeu uma borda visual inspirada no cordão TEA, reforçando a relação do projeto com inclusão, acessibilidade e educação inclusiva.

    - A organização do título, subtítulo, selo institucional e seleção de perfil foi ajustada para deixar a tela inicial mais harmônica e apresentável.


- Personalização do chatbot

    - O componente do chatbot foi estilizado para parecer mais limpo, moderno e centralizado dentro da interface.

    - Foi adicionada uma marca d’água da Accenture no fundo do chatbot, de forma discreta, sem atrapalhar a leitura das mensagens.

    - Foram ajustadas as fontes, espaçamentos, altura do chat, bordas e sombras para tornar a leitura mais confortável.

    - Também foi feita uma correção visual para evitar a sensação de vários cards dentro do chat, deixando as mensagens mais limpas e sem blocos duplicados.


- Criação e ajuste do dropdown de perfil

    - Foi implementado e estilizado o dropdown de tipo de usuário, permitindo selecionar entre os perfis Professor, Família e Gestor.

    - O dropdown foi reposicionado no cabeçalho para ficar mais integrado ao layout e não parecer um elemento solto na tela.

    - Foram removidos efeitos visuais duplicados do próprio Gradio, deixando o dropdown com apenas um box visual mais limpo.

    - Também foi ajustado o comportamento da lista de opções, garantindo que ela apareça por cima dos outros elementos da tela e não fique escondida atrás de cards ou campos de texto.

    - A escolha do perfil passou a fazer parte da experiência visual do sistema, reforçando que o assistente pode se adaptar a diferentes tipos de usuários.


- Botões e interações visuais

    - Os botões principais, como “Enviar pergunta” e “Limpar histórico”, foram personalizados.

    - No estado padrão, os botões ficam brancos com texto roxo, mantendo um visual limpo e profissional.

    - Ao passar o cursor, os botões mudam para fundo roxo com texto branco, deixando a interação mais clara para o usuário.

    - Foi ajustada a velocidade do hover para que a troca de cor ficasse mais rápida, natural e sem fazer o texto desaparecer.

    - As perguntas sugeridas também receberam estilo próprio, com botões mais organizados, melhor espaçamento e interação visual ao passar o mouse.


- Cards laterais e informações complementares

    - Foi criado um card lateral explicando o objetivo do protótipo.

    - Também foi criada uma área de perguntas sugeridas, ajudando o usuário a entender como interagir com o assistente.

    - Os cards receberam bordas, sombras leves e melhor padronização visual para combinar com o restante da interface.

    - A distribuição dos cards foi ajustada para deixar a tela mais equilibrada e facilitar a navegação do usuário.


- Implementação do painel de fontes consultadas

    - Foi adicionado um card específico para exibir as fontes consultadas pelo assistente.

    - Esse painel mostra os artigos mais relevantes usados como base para a resposta, incluindo título, link e pontuação de relevância.

    - Essa funcionalidade melhora a transparência do protótipo, pois mostra que as respostas são baseadas em fontes reais e não apenas em respostas geradas livremente pela inteligência artificial.

    - O painel de fontes também contribui para dar mais credibilidade ao projeto durante a apresentação para a banca.


- Responsividade e acabamento visual

    - Foram feitos ajustes para que a interface também funcione melhor em telas menores.

    - Em telas abaixo de 900px, o layout se adapta, reorganizando os elementos em coluna e ajustando tamanhos de título, cards e chatbot.

    - Também foram personalizados detalhes como barra de rolagem, rodapé, espaçamentos e proporções gerais da página.

    - Esses ajustes deixam o protótipo mais preparado para diferentes resoluções de tela, melhorando a experiência de uso.


- Construção da interface com Gradio na Sessão 10

    - Na Sessão 10, foi construída a interface principal do protótipo usando Gradio.

    - Essa seção organiza a parte visual do sistema, conectando o chatbot, o campo de pergunta, os botões, o dropdown de perfil, os cards laterais e o painel de fontes.

    - O CSS foi carregado externamente pelo notebook, deixando o código Python mais limpo e separando melhor a parte visual da parte lógica.

    - A Sessão 10 ficou responsável por reunir os principais elementos visuais e funcionais da aplicação em uma única interface interativa.


- Carregamento de imagens no front

    - Foram carregadas imagens do projeto, como a logo da Accenture, o cordão TEA, o ícone do grupo e a logo do DIVERSA.

    - Essas imagens foram convertidas para Base64, permitindo que fossem usadas diretamente no HTML e no CSS da interface.

    - Isso ajudou a deixar o front mais personalizado e alinhado com a identidade visual pensada para o TCC.

    - O uso dessas imagens reforçou visualmente a proposta do projeto, relacionando tecnologia, inclusão e educação.


- Histórico separado por perfil

    - Foi implementada uma lógica para manter históricos separados para cada tipo de usuário: Professor, Família e Gestor.

    - Ao trocar de perfil, o sistema salva a conversa anterior e carrega o histórico correspondente ao novo perfil.

    - Isso evita que perguntas e respostas de perfis diferentes se misturem, deixando a experiência mais organizada e personalizada.

    - Essa funcionalidade foi pensada para respeitar o contexto de cada usuário, já que cada perfil pode ter dúvidas e necessidades diferentes.


- Integração entre perfil e resposta do assistente

    - O dropdown de perfil foi conectado à lógica do chatbot.

    - Dependendo do perfil selecionado, o sistema pode adaptar o comportamento da resposta, considerando o contexto do usuário.

    - Essa implementação reforça a proposta do projeto de oferecer uma experiência mais adequada para diferentes públicos.

    - A escolha entre Professor, Família e Gestor permite que o assistente tenha uma abordagem mais direcionada e coerente com cada perfil.


- Botões funcionais da interface

    - Foram configurados os eventos dos botões principais da aplicação.

    - O botão “Enviar pergunta” envia a mensagem digitada pelo usuário para o assistente.

    - O botão “Limpar histórico” limpa apenas a conversa do perfil selecionado, sem apagar os históricos dos outros perfis.

    - As perguntas sugeridas preenchem automaticamente o campo de texto, facilitando o uso do sistema por usuários que não sabem por onde começar.

    - Essa funcionalidade melhora a usabilidade e torna o protótipo mais acessível para diferentes públicos.


## Requisito 8 - Cesar ou Nicolas

**8. Ajustar os parâmetros do modelo por perfil de usuário**

  - Objetivo da implementação

    - Foi realizada uma adaptação na lógica do assistente para que o comportamento do modelo pudesse variar de acordo com o perfil de usuário selecionado.

    - A ideia foi permitir que o assistente respondesse de forma mais adequada para diferentes públicos, considerando que Professor, Família e Gestor possuem necessidades, dúvidas e níveis de linguagem diferentes.

    - Essa funcionalidade foi pensada para tornar a experiência mais personalizada, evitando que todos os usuários recebessem respostas com o mesmo tom, profundidade e direcionamento.

    - O ajuste dos parâmetros por perfil contribui para que o chatbot fique mais coerente com a proposta de inclusão, acessibilidade e educação inclusiva do projeto.

- Perfis de usuário considerados

    - Foram definidos três perfis principais de uso: Professor, Família e Gestor.

    - O perfil Professor foi pensado para usuários que precisam de respostas mais pedagógicas, com orientações práticas para sala de aula, adaptação de atividades e estratégias de inclusão.

    - O perfil Família foi pensado para responsáveis, pais ou familiares que precisam de respostas mais acolhedoras, simples e explicativas.

    - O perfil Gestor foi pensado para usuários que precisam de respostas mais institucionais, voltadas para organização escolar, políticas de inclusão, planejamento e tomada de decisão.

- Ajuste do comportamento do modelo

    - Para cada perfil, foi pensada uma forma diferente de conduzir a resposta do assistente.

    - O modelo pode adaptar o tom da resposta, o nível de detalhamento e o tipo de orientação oferecida.

    - No perfil Professor, as respostas tendem a ser mais objetivas e aplicáveis ao contexto pedagógico.

    - No perfil Família, as respostas priorizam uma linguagem mais simples, acolhedora e de fácil compreensão.

    - No perfil Gestor, as respostas são mais formais, organizadas e voltadas para decisões administrativas e institucionais.

- Personalização do prompt por perfil

    - Foi criada uma lógica para que o perfil selecionado influencie o contexto enviado ao modelo.

    - Isso significa que, antes de gerar a resposta, o sistema considera qual tipo de usuário está utilizando o assistente.

    - A partir dessa escolha, o modelo passa a responder com uma abordagem mais adequada para aquele público.

    - Essa personalização ajuda a evitar respostas genéricas e melhora a qualidade da interação com o usuário.

- Importância para a experiência do usuário

    - A implementação melhora a usabilidade do sistema, pois cada usuário recebe uma resposta mais próxima da sua realidade.

    - Um professor pode receber orientações práticas para aplicar em sala de aula.

    - Uma família pode receber explicações mais claras sobre inclusão, direitos e formas de apoio.

    - Um gestor pode receber respostas mais voltadas para planejamento, acessibilidade institucional e organização escolar.

    - Isso torna o assistente mais útil, pois ele não trata todos os usuários da mesma forma.

- Relação com a proposta de inclusão

    - O ajuste dos parâmetros por perfil está diretamente ligado à proposta inclusiva do projeto.

    - Como o sistema atende públicos diferentes, foi necessário pensar em uma comunicação mais acessível e personalizada.

    - A inclusão não está apenas no conteúdo das respostas, mas também na forma como o sistema se comunica com cada usuário.

    - Dessa maneira, o assistente se torna mais sensível ao contexto de quem está utilizando a ferramenta.

- Benefícios da funcionalidade

    - A funcionalidade torna o chatbot mais inteligente e adaptável.

    - Melhora a clareza das respostas.

    - Reduz respostas genéricas.

    - Aumenta a sensação de personalização durante o uso.

    - Organiza melhor a experiência de acordo com o tipo de usuário.

    - Contribui para uma navegação mais simples e direcionada.

    - Fortalece a proposta acadêmica do projeto ao mostrar que houve preocupação com diferentes públicos.

- Integração com o dropdown de perfil

    - O ajuste dos parâmetros foi integrado ao dropdown de seleção de usuário.

    - Quando o usuário escolhe Professor, Família ou Gestor, essa informação passa a influenciar o funcionamento do assistente.

    - O dropdown não funciona apenas como um elemento visual, mas também como parte da lógica de personalização do sistema.

    - Isso torna a interface mais funcional, pois a escolha do perfil interfere diretamente na resposta gerada.

- Exemplo de aplicação prática

    - Se uma pergunta sobre inclusão escolar for feita no perfil Professor, o assistente pode responder com sugestões de atividades, adaptações e estratégias pedagógicas.

    - Se a mesma pergunta for feita no perfil Família, o assistente pode explicar o tema com uma linguagem mais simples e acolhedora.

    - Se a pergunta for feita no perfil Gestor, o assistente pode trazer uma resposta mais voltada para organização da escola, políticas de acessibilidade e ações institucionais.

    - Dessa forma, a mesma base de conhecimento pode ser usada de maneiras diferentes, dependendo do perfil selecionado.

- Desafio encontrado

    - Um dos desafios foi pensar em como diferenciar as respostas sem alterar totalmente a estrutura principal do chatbot.

    - Também foi necessário garantir que a troca de perfil não deixasse a experiência confusa para o usuário.

    - A solução foi vincular o perfil selecionado ao contexto da resposta, mantendo a interface simples, mas com uma lógica mais personalizada por trás.

    - Isso permitiu que o assistente continuasse fácil de usar, mas com respostas mais direcionadas.

- Resultado alcançado

    - Com essa implementação, o sistema passou a oferecer uma experiência mais personalizada.

    - O assistente deixou de responder de forma única para todos os usuários e passou a considerar o perfil selecionado.

    - Isso melhorou a coerência das respostas e reforçou a proposta do projeto.

    - O resultado foi um chatbot mais adaptável, organizado e adequado para diferentes contextos de uso.
  

## Requisito 9 - Cesar


**9. Implementar histórico de conversa**

- Objetivo da implementação

    - Foi implementada uma lógica de histórico de conversa dentro do protótipo, com o objetivo de manter as interações do usuário organizadas durante o uso do assistente.

    - A funcionalidade foi pensada para permitir que o usuário acompanhe melhor o andamento da conversa com o chatbot.

    - Antes dessa implementação, as mensagens poderiam se misturar ou se perder ao trocar de contexto, dificultando a continuidade do atendimento.

    - Com o histórico de conversa, o sistema passou a preservar as perguntas e respostas feitas durante a interação, tornando a experiência mais clara e organizada.

- Organização da conversa na interface

    - O histórico foi integrado diretamente ao componente de chat da interface.

    - Cada mensagem enviada pelo usuário e cada resposta gerada pelo assistente passam a ser exibidas em sequência.

    - Isso permite que o usuário visualize todo o diálogo, sem precisar lembrar exatamente o que perguntou anteriormente.

    - A organização em formato de conversa torna o uso mais natural, semelhante a outras ferramentas de chatbot já conhecidas.

- Preservação do contexto da conversa

    - A implementação do histórico ajuda o assistente a manter o contexto da interação.

    - Quando o usuário faz uma nova pergunta, o sistema pode considerar o que já foi conversado anteriormente.

    - Isso evita respostas desconectadas e melhora a continuidade do atendimento.

    - A preservação do contexto torna o chatbot mais útil, pois a conversa não fica limitada apenas à pergunta atual.

- Histórico separado por perfil de usuário

    - O histórico foi pensado também para funcionar de forma organizada conforme o perfil selecionado.

    - Como o sistema trabalha com diferentes perfis, como Professor, Família e Gestor, foi necessário evitar que as conversas desses usuários se misturassem.

    - Ao trocar de perfil, o sistema preserva o histórico anterior e carrega a conversa correspondente ao novo perfil selecionado.

    - Dessa forma, cada perfil mantém sua própria linha de conversa, respeitando o contexto e a necessidade de cada tipo de usuário.

- Relação com os perfis Professor, Família e Gestor

    - O perfil Professor pode manter conversas voltadas para práticas pedagógicas, adaptação de atividades e estratégias para sala de aula.

    - O perfil Família pode manter conversas com dúvidas mais simples, explicações acessíveis e orientações de apoio.

    - O perfil Gestor pode manter conversas mais institucionais, relacionadas à organização escolar, acessibilidade e planejamento.

    - Separar o histórico por perfil ajuda a preservar essas diferenças e evita que uma conversa interfira na outra.

- Melhoria na experiência do usuário

    - A implementação melhora a usabilidade do sistema, pois o usuário consegue acompanhar melhor o que já foi perguntado e respondido.

    - O histórico evita repetições desnecessárias, já que a conversa anterior continua visível.

    - Também facilita a continuidade da interação, permitindo que o usuário aprofunde um assunto sem precisar começar do zero.

    - Essa funcionalidade torna o sistema mais intuitivo, organizado e próximo de uma experiência real de atendimento por inteligência artificial.

- Botão de limpar histórico

    - Foi implementada a função de limpar o histórico da conversa.

    - Essa opção permite que o usuário apague as mensagens quando quiser iniciar uma nova conversa.

    - A função de limpar histórico ajuda a manter a interface organizada e evita acúmulo de mensagens desnecessárias.

    - No caso do histórico por perfil, a limpeza pode ser aplicada ao perfil selecionado, sem necessariamente apagar os históricos dos outros perfis.

- Integração com a interface do chatbot

    - O histórico foi integrado ao campo principal de conversa do Gradio.

    - A cada nova pergunta, o conteúdo digitado pelo usuário é enviado ao assistente e a resposta é adicionada ao histórico.

    - O chat passa a funcionar de forma dinâmica, atualizando a tela conforme a conversa acontece.

    - Essa integração torna a experiência mais fluida e facilita a visualização da interação completa.

- Importância para o projeto

    - O histórico de conversa é uma parte importante do protótipo, pois torna o assistente mais funcional e próximo de um sistema real.

    - Sem o histórico, a experiência ficaria limitada a perguntas isoladas.

    - Com o histórico, o usuário consegue desenvolver uma conversa contínua com o assistente.

    - Essa funcionalidade contribui para a credibilidade do projeto, mostrando que o chatbot foi pensado não apenas para responder perguntas, mas também para oferecer uma experiência organizada de atendimento.

- Relação com a proposta de inclusão

    - A implementação do histórico também se relaciona com a proposta inclusiva do projeto.

    - Uma conversa organizada facilita o uso por diferentes públicos, inclusive pessoas que precisam reler informações ou acompanhar o raciocínio com mais calma.

    - O histórico ajuda a tornar a navegação mais acessível, pois permite que o usuário consulte novamente respostas anteriores.

    - Isso é importante em um projeto voltado para educação inclusiva, já que clareza, organização e continuidade são elementos fundamentais para melhorar a experiência de uso.

- Desafios encontrados

    - Um dos desafios foi controlar a troca de perfis sem perder as mensagens anteriores.

    - Também foi necessário garantir que cada perfil mantivesse sua própria conversa durante o uso do sistema.

    - Outro ponto importante foi permitir que o usuário limpasse o histórico sem comprometer toda a lógica da aplicação.

    - A solução foi criar uma lógica de armazenamento das conversas por perfil, permitindo salvar, recuperar e limpar o histórico de forma organizada.

- Resultado alcançado

    - Com essa implementação, o chatbot passou a oferecer uma experiência mais completa e organizada.

    - O usuário consegue visualizar suas perguntas e as respostas do assistente em sequência.

    - A troca de perfil ficou mais estruturada, pois cada perfil pode manter sua própria conversa.

    - O resultado foi uma interface mais funcional, intuitiva e adequada para a proposta do TCC.

## Requisito 10 - Nicolas


**10. Adicionar métricas de uso**
  Registrar em um DataFrame: data/hora, pergunta, artigos recuperados e score de
  similaridade. Ao final de um período, gerar um relatório com os temas mais
  perguntados e a qualidade média das buscas.

## Requisito 11 - JP


**11. Experimentar embeddings semânticos**
  O TF-IDF não entende sinônimos — "criança cega" e "deficiência visual" são
  tratados como temas diferentes. Uma evolução natural é usar embeddings semânticos
  com modelos multilíngues gratuitos que entendem o significado
  das palavras, não apenas a forma.

## Requisito 12 - JP


**12. Avaliar outros modelos gratuitos da Groq**
  Testar outros modelos também gratuitos na Groq, Llama, Hugging Face etc.)
  e comparar a qualidade das respostas em português com o modelo atual.
  Documentar a diferença de velocidade e qualidade.

## Ideias mirabolantes

Quais dessas ideias poderiam ser realidade?

1. Arquitetura Llama: 3.1 8B (raciocínio do agente) + GraphRAG (conexão entre os artigos) + Reranking (filtro de relevância)
2. Assim que acabar o requisito 12, implantar fallback para o segundo agente do ranking quando o modelo atingir o limite de requisições diárias, ou para os modelos subsequentes do ranking de velocidade x qualidade.
